<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec01_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec01_intro.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第1回 導入と全体概要
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib，PyWavelets

MATLAB のライブスクリプト `code/matlab/vie_sec01_intro.m` を Python に移植したもの。第1回のスライド（vie2026-01）の図と数値を計算し，スライドに書ききれない計算の途中経過も丁寧に追いかける。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import os
import urllib.request

if not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")
try:
    import pywt
except ImportError:
    %pip install -q PyWavelets
    import pywt

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from skimage.util import img_as_float
import vie

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

## 音声信号：一変量の関数とその標本化
音叉の音のように，音声信号（モノラル）は時刻 $t$ の**一変量の関数** $u(t)$ とみなせる（参考資料 1.4.1 項の記法）。ここでは 1 秒間に 440 回振動する音（ラの音）を，少しずつ減衰する正弦波で模擬する。

$$u(t) = \mathrm{e}^{-t/\tau}\sin(2\pi f_0 t),\quad f_0 = 440\ \mathrm{Hz}$$

これを標本化周波数 $f_\mathrm{s}$（標本化間隔 $\Delta_\mathrm{t}=1/f_\mathrm{s}$）で標本化すると，**数列（サンプル列）** $x[n] = u(\Delta_\mathrm{t}n)$ が得られる。

In [ ]:
f0 = 440     # 音の高さ [Hz]
tau = 0.4    # 減衰の時定数 [s]
fs = 8000    # 標本化周波数 [Hz]
n = np.arange(fs)                      # 1 秒分の標本番号 n = 0,1,...,fs-1
xn = np.exp(-n / fs / tau) * np.sin(2 * np.pi * f0 * n / fs)

冒頭の 5 ms だけを拡大すると，連続な波形 $u(t)$（灰色）の上に標本 $x[n]$（緑）が等間隔に並んでいるのが分かる。右は周波数成分の時間変化（スペクトログラム）で，440 Hz に成分が集中している。「どのような成分から構成されているか」を調べるのがフーリエ解析（第7回）である。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3), layout="constrained")
tc = np.linspace(0, 5e-3, 500)
axs[0].plot(tc * 1e3, np.exp(-tc / tau) * np.sin(2 * np.pi * f0 * tc), color=cgray, lw=1)
ns = np.arange(round(5e-3 * fs) + 1)
axs[0].stem(ns / fs * 1e3, xn[ns], linefmt="-", markerfmt="o", basefmt=" ")
axs[0].set(xlabel="時刻 $t$ [ms]", ylabel="振幅", title="$u(t)$ と標本 $x[n]$")
axs[0].grid(True)
# スペクトログラム（窓長 256 のハン窓，重なり 192，FFT 点数 512）
f, t, S = signal.spectrogram(xn, fs, window="hann", nperseg=256, noverlap=192, nfft=512)
axs[1].pcolormesh(t * 1e3, f / 1e3, 10 * np.log10(S + 1e-12), shading="auto")
axs[1].set(ylim=(0, 2), xlabel="時刻 $t$ [ms]", ylabel="周波数 [kHz]", title="スペクトログラム")
plt.show()

## 画像信号：多変量の関数とその標本化
画像信号（モノクロ）は位置 $(q_\mathrm{v},q_\mathrm{h})$ の**多変量の関数** $u(q_\mathrm{v},q_\mathrm{h})$ とみなせる。標本化すると**配列（画素配列）** $x[n_\mathrm{v},n_\mathrm{h}]$ が得られる。

画像は参考資料のサンプル画像 msipimg05（石像の顔，512×512 のカラー）を 256×256 に縮小し，グレースケールにしたものを使う。

In [ ]:
X = vie.msipimg(5, 256, "gray")   # 8 bit グレースケール画像（uint8）
print("大きさ:", X.shape, " 型:", X.dtype)

一配列要素あたりのビット数 $\beta$ は，配列の占めるバイト数（`nbytes`）を要素数で割れば分かる（MATLAB 版の `whos` と同じ確かめ方）。

In [ ]:
bitsX = 8 * X.nbytes / X.size    # 8 bits（uint8 型）
print("β =", bitsX, "bits")

画像の一部（左上の空と石像の境目）を切り出して数値を見る（縮小の実装が MATLAB と異なるので，値はスライドとわずかに異なる）。明るい空（200 台）から暗い石（100 前後以下）へ，値が斜めに切り替わる様子が数値で分かる。画像は数の並びにすぎない。Python の添字は教科書の配列 $x[n_\mathrm{v},n_\mathrm{h}]$ と同じく 0 から始まる。

In [ ]:
r0, c0, w = 49, 26, 5            # 切り出す位置 (n_v, n_h) と大きさ
blk = X[r0:r0 + w, c0:c0 + w]
print(f"n_v = {r0}〜{r0 + w - 1}, n_h = {c0}〜{c0 + w - 1}")
print(blk)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
show(axs[0], X, f"{X.shape[0]}×{X.shape[1]} 画素")
axs[0].add_patch(plt.Rectangle((c0 - 0.5, r0 - 0.5), w, w, fill=False, ec=cwarm, lw=1.5))
show(axs[1], blk, "橙枠内の画素値")
for i in range(w):
    for j in range(w):
        axs[1].text(j, i, str(blk[i, j]), ha="center", va="center", color=cwarm, fontsize=10)
plt.show()

## 信号の解析（一次元）：近似成分と詳細成分
前回スライドの例を計算する。信号 $x[n]\ (n=0,1,2,\ldots)$ に対し，**隣同士を足して 2 で割る**と近似成分 $a[n]$，**左隣から引いて 2 で割る**と詳細成分 $d[n]$ が得られる。

現在と過去の標本だけを使う**因果的**な形に書く。こうすると，次節のフィルタ $H_0(z)=\frac{1}{2}(1+z^{-1})$，$H_1(z)=\frac{1}{2}(-1+z^{-1})$ の出力そのものになる。

$$a[n] = \frac{1}{2}\left(x[n]+x[n-1]\right),\qquad d[n] = \frac{1}{2}\left(-x[n]+x[n-1]\right),\qquad n=1,2,\ldots$$

In [ ]:
x = np.array([3, 1, 3, 1, 5, 3, 5])   # x[0], x[1], ...
a = (x[1:] + x[:-1]) / 2              # 近似成分 a[n]，n = 1,2,...
d = (-x[1:] + x[:-1]) / 2             # 詳細成分 d[n]，n = 1,2,...
print("a =", a)
print("d =", d)

次節の $H_0(z)$，$H_1(z)$ で x をフィルタリングした出力（n=1 以降）と一致することを確かめる。近似成分はゆっくり変わる成分（低周波成分），詳細成分は細かく変わる成分（高周波成分）を担う。

In [ ]:
a_f = signal.lfilter([1 / 2, 1 / 2], 1, x)
d_f = signal.lfilter([-1 / 2, 1 / 2], 1, x)
assert np.array_equal(a, a_f[1:]) and np.array_equal(d, d_f[1:])
# スライドの値と一致することを確かめる
assert np.array_equal(a, [2, 2, 2, 3, 4, 4]) and np.array_equal(d, [1, -1, 1, -2, 1, -1])

## 信号の合成：足し合わせると元に戻る
定義から $a[n]+d[n] = x[n-1]$ が成り立つ。1 標本遅れるだけで元に戻る（全体の伝達関数 $T(z)=H_0(z)+H_1(z)=z^{-1}$）。足し算と引き算だけの分解だが，**信号の情報を逃さない**立派な成分分析法である。

In [ ]:
xr = a + d
print("a + d =", xr)
assert np.array_equal(xr, x[:-1])    # x[n-1]，n = 1,2,...

近似成分を緑，詳細成分を青，両者の和を橙で描く。和の図には元の信号 $x[n]$ を灰色の白抜きの丸で重ね，一致することを示す。

In [ ]:
nn = np.arange(1, len(a) + 1)        # n = 1,2,...
fig, axs = plt.subplots(1, 3, figsize=(10, 2.8), layout="constrained")
for ax, v, c, ttl in zip(axs, (a, d, xr), (cmain, ccool, cwarm),
                         ("近似成分 $a[n]$", "詳細成分 $d[n]$", "$a[n]+d[n]$（○：$x[n-1]$）")):
    ml, sl, bl = ax.stem(nn, v, basefmt="k-")
    plt.setp(ml, color=c)
    plt.setp(sl, color=c)
    ax.set(ylim=(-3, 6), xlabel="$n$", title=ttl)
    ax.grid(True)
axs[2].plot(nn, x[:-1], "o", mfc="none", color=cgray, ms=10, mew=1.2)
plt.show()

## フィルタバンク表現
上の操作はフィルタで書ける。遅延器 $z^{-1}$ を使うと，近似成分と詳細成分を作るフィルタの伝達関数は

$$H_0(z) = \frac{1}{2}\left(1+z^{-1}\right),\qquad H_1(z) = \frac{1}{2}\left(-1+z^{-1}\right)$$

両者を足すと $T(z) = H_0(z)+H_1(z) = z^{-1}$，すなわち**全体は単なる遅延**になる。インパルス応答（係数列）で確かめる。

In [ ]:
h0 = np.array([1, 1]) / 2     # H0(z) の係数（z^0, z^-1 の順）
h1 = np.array([-1, 1]) / 2    # H1(z) の係数
t = h0 + h1
print("T(z) の係数:", t, "（[0 1] は z^{-1}）")

実際に信号を通すと，出力は入力を 1 標本遅らせたものになる。

In [ ]:
y = np.convolve(x, h0) + np.convolve(x, h1)
print("y =", y)

## 画像変換の効果：3 レベルウェーブレット変換
画像にも同じ考え方（近似と詳細への分解）を縦横に繰り返し適用できる。ここではハールウェーブレットで 3 レベル分解する。

画像は参考資料のサンプル画像 msipimg08（スイカ）を 256×256 のグレースケールにしたものを使う。縞模様の輪郭にだけ大きな係数が残り，それ以外はほぼ零になる様子がよく見える。

In [ ]:
Xd = img_as_float(vie.msipimg(8, 256, "gray"))
coeffs = pywt.wavedec2(Xd, "haar", level=3)   # [A3, (H3,V3,D3), (H2,V2,D2), (H1,V1,D1)]
C = np.concatenate([coeffs[0].ravel()] + [b.ravel() for lv in coeffs[1:] for b in lv])

変換前の画素値は 0 から 1 まで広く分布する**デンス（密）**な表現だが，変換後の係数は**ほとんどが零に近い**。

In [ ]:
thr = 0.05                               # 「ほぼ零」とみなすしきい値
pz = np.mean(np.abs(C) < thr) * 100      # ほぼ零の係数の割合 [%]
px = np.mean(Xd < thr) * 100             # 比較：画素値が thr 未満の割合 [%]
print(f"ほぼ零の係数 {pz:.1f} %，画素値が {thr} 未満 {px:.1f} %")

係数を画像として並べる。左上に小さく残るのが近似成分，残りが各レベル・各方向の詳細成分である（詳細成分は見やすいよう 2 倍して表示）。

In [ ]:
coef = np.zeros_like(Xd)
A3 = coeffs[0]
n3 = A3.shape[0]
coef[:n3, :n3] = A3 / A3.max()
for H, V, D in coeffs[1:]:              # レベル 3, 2, 1 の順
    m = H.shape[0]
    coef[:m, m:2 * m] = np.abs(H) * 2
    coef[m:2 * m, :m] = np.abs(V) * 2
    coef[m:2 * m, m:2 * m] = np.abs(D) * 2
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
show(axs[0], Xd, "変換前（デンス）")
show(axs[1], np.minimum(coef, 1), "変換後（ほとんど零）")
plt.show()

## データ量：静止画像
教科書の例「静止画像のデータ量」（1.1.2 項）。$N_1\times N_2$ 画素，一配列要素あたり $\beta$ ビットのグレースケール画像の総ビット数は $B = \beta N_1 N_2$。RGB カラー画像なら一画素あたり $3\beta$ [bpp] なので $B = 3\beta N_1 N_2$。

In [ ]:
N1, N2 = 2304, 3456               # 画素数
Npix = N1 * N2                    # 総画素数（約 800 万画素）
bt = 8                            # 8-bit 符号なし整数型
Bgray = bt * N1 * N2              # グレースケール画像 [bits]
bt = 64                           # 倍精度実数型
Brgb = 3 * bt * N1 * N2           # RGB カラー画像 [bits]
ratio = Brgb // Bgray             # 後者は前者の何倍か
print(f"総画素数 {Npix:,}，グレースケール {Bgray:,} bits，RGB {Brgb:,} bits，{ratio} 倍")

教科書の値（63700992 bits ≃ 8 MB，1528823808 bits ≃ 191 MB，24 倍）と一致することを確かめる。Mega は $10^6$，1 byte は 8 bits とする。

In [ ]:
assert (Bgray, Brgb, ratio) == (63700992, 1528823808, 24)
print(f"{Bgray / 8 / 1e6:.0f} MB，{Brgb / 8 / 1e6:.0f} MB")

実際の画像データでも確かめる。参考資料のサンプル画像 msipimg02（花束，512×512）は 8-bit の RGB カラー画像で，倍精度実数型に変換すると 8 倍のバイト数になる。

In [ ]:
P = vie.msipimg(2)                # uint8 の RGB カラー画像（512×512×3）
Pd = img_as_float(P)             # 倍精度実数型に変換
N1p, N2p = P.shape[:2]
print("β = 8 :", P.nbytes, "bytes ／ 3βN1N2/8 =", 8 * 3 * N1p * N2p // 8)
print("β = 64:", Pd.nbytes, "bytes ／ 3βN1N2/8 =", 64 * 3 * N1p * N2p // 8)

## 1 秒あたりのビット数（ビットレート）
動画像では **画素数 × 画面数/秒 × ビット数/画素** がビットレート $R$ [bps] になる。教科書の式 $R = \beta|\Omega_\mathrm{S}||\Omega_\mathrm{C}|\Delta_\mathrm{t}^{-1}$ で，RGB 各 8 bit なら一画素 24 bit である。まず前回スライドの HDTV と SDTV（いずれも 30 フレーム/秒）を計算する。

In [ ]:
Rhd = 1920 * 1080 * 30 * 24       # ハイビジョン品質（HDTV）[bps]
Rsd = 720 * 480 * 30 * 24         # アナログ放送品質（SDTV）[bps]
print(f"HDTV {Rhd / 1e9:.1f} Gbps，SDTV {Rsd / 1e6:.1f} Mbps")

### 教科書の例題「動画像のビットレート」
画素数 $N_1\times N_2 = 4320\times 7680$（8K），フレーム間隔 $\Delta_\mathrm{t} = 1/60$ s，一配列要素あたり $\beta = 8$ bits の RGB カラー動画像のビットレートを求める。フレームレートは $\Delta_\mathrm{t}^{-1} = 60$ [1/s] である（$1/60$ は二進の浮動小数点数で割り切れないので，逆数のまま掛ける）。

In [ ]:
N1, N2 = 4320, 7680               # 画素数
bt = 8                            # 一配列要素あたりのビット数
fps = 60                          # フレームレート 1/Δt [1/s]
bpp8k = 3 * bt                    # 一画素あたりのビット数 [bpp]
N8k = N1 * N2                     # 1 フレームあたりの画素数（約 3300 万画素）
B8k = 3 * bt * N1 * N2            # 1 フレームあたりのビット数 [bits/frame]
R8k = B8k * fps                   # ビットレート R = B Δt^{-1} [bps]
print(f"{bpp8k} bpp，{N8k:,} 画素，{B8k:,} bits/frame，{R8k:,} bps ≃ {R8k / 1e9:.0f} Gbps")

教科書の解答（24 bpp，33177600 画素，796262400 bits/frame，47775744000 bps ≃ 48 Gbps）と一致することを確かめる。実際に 1 フレーム分の配列を作って要素数から数えても同じになる。

In [ ]:
assert (bpp8k, N8k, B8k, R8k) == (24, 33177600, 796262400, 47775744000)
frame = np.zeros((N1, N2, 3), dtype=np.uint8)   # 8K の RGB カラー 1 フレーム（約 100 MB）
assert bt * frame.size == B8k
del frame

### 地上デジタル放送との比較
地上デジタル放送では HDTV を約 14 Mbps，SDTV を約 4 Mbps で送る（前回スライドの値）。圧縮前と比べると次のとおり。

In [ ]:
Rdtv = np.array([14e6, 4e6])
cr = np.array([Rhd, Rsd]) / Rdtv   # 何分の一に圧縮しているか
print(f"HDTV 1/{cr[0]:.0f}，SDTV 1/{cr[1]:.0f}")

圧縮前を青，地デジを緑の横棒で比べる。棒の右端に値を添える。

In [ ]:
labels = ["SDTV 地デジ", "SDTV 圧縮前", "HDTV 地デジ", "HDTV 圧縮前"]
vals = np.array([Rdtv[1], Rsd, Rdtv[0], Rhd]) / 1e6   # [Mbps]
fig, ax = plt.subplots(figsize=(7, 2.6), layout="constrained")
ax.barh(np.arange(4), vals, 0.6, color=[cmain, ccool, cmain, ccool])
for i, v in enumerate(vals):
    ax.text(v + 30, i, f"{v:.0f} Mbps", va="center")
ax.set(yticks=np.arange(4), yticklabels=labels, xlim=(0, 2100), xlabel="ビットレート [Mbps]")
ax.grid(True, axis="x")
plt.show()

## まとめ
- 音声は一変量の関数，画像は多変量の関数。標本化すると数列・配列になる
- 足し算と引き算だけで，信号を近似成分と詳細成分に分解・合成できる
- 画像を変換すると，係数のほとんどが零に近くなる（スパースな表現）
- データ量は $B=\beta N_1N_2$（RGB なら $3\beta N_1N_2$），ビットレートは $R=B\Delta_\mathrm{t}^{-1}$
- 非圧縮の映像は Gbps 級。放送では 1/100 程度に圧縮している

© Copyright, Shogo MURAMATSU, All rights reserved.